In [1]:
import numpy as np
import cvxpy as cp
import mosek
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

In [10]:
def makeset (A, B):    # we assume A is non-empty
    N = len(A)
    added = []
    for i in range(N):
        new = B[0:i+1]
        N_sets = len(A[i])
        for k in range(N_sets-1):
            if len(np.intersect1d(A[i][k],new))==len(new):
                break
            if k == N_sets-2:
                A[i].append(new)
                added.append(new)
    return(A,added)

def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)

def solvenominal (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] <= 0)
        constraints.append((-R @ a)[i] - lbdasum <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    
    constraints.append(alpha + gamma * r - (1-cp.sum(a))*r_f + z4 -1 + z2 <= c)
    constraints.append(a<=10)
    constraints.append(a>=-10)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value,v.value,a.value)
    
    
def robustcheck(a,R,r,c,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.min(x) < 0:
        extra = np.min(x)
        c = c - np.min(x)
        x = x - np.min(x)
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -(cp.entr(q[i]) - q[i]*np.log(p[i]))
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    print(c)
    print(prob.value - (1-np.sum(a))*r_f+ extra)
    return(prob.value,q.value,q_b.value,(prob.value - (1-np.sum(a))*r_f+ extra) <= c)
    
                
    

In [4]:
N=4
x=np.array([1,2,3])
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
psets

[[0],
 [1],
 [2],
 [3],
 [0, 1],
 [0, 2],
 [0, 3],
 [1, 2],
 [1, 3],
 [2, 3],
 [0, 1, 2],
 [0, 1, 3],
 [0, 2, 3],
 [1, 2, 3],
 [0, 1, 2, 3]]

In [5]:
r = 1
m = 0.2
r_f = 0.001
c = 0.001
p = np.array([0.2,0.4,0.3,0.1])
R = np.array([[0.01,-0.5,0.5,0.55,-0.6],[0.1,0.42,-0.38,-0.36,0.51],[-0.53,0.57,-0.58,-0.96,-0.62],
              [-0.02,-0.75,0.33,0.09,0.573]])
sets =psets               #[[0],[1],[0,1],[1,3],[0,3],[0,1,2],[1,3,2],[0,1,3],[0,1,2,3]]
solvenominal (sets,p,R,r,m,r_f,c)

(8.367999999999997,
 array([[ 12.        ,  -1.83279643,  -1.83279643, -23.37029643],
        [-13.83279643,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  21.5375    ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0.        ],
        [ -0.        ,  -0.        ,  -0.        ,  -0. 

In [6]:
a=solvenominal (sets,p,R,r,m,r_f,c)[2]           #np.array([1/5,1/5,1/5,1/5,1/5])
print(R.dot(a).dot(p))
print(-R.dot(a))

8.336999999999996
[  9.6   -5.5  -32.6   17.23]


In [11]:
robustcheck(a,R,r,c,p,m,r_f)

32.60099999999999
17.19899999999847


(49.82999999999846,
 array([0.0422333 , 0.04850805, 0.077579  , 0.83167965]),
 array([ 1.14471692e-13,  1.86827328e-14, -0.00000000e+00,  1.00000000e+00]),
 True)

In [242]:
np.min([1,0.83167965/(1-0.2)])

1.0

In [253]:
(1-np.sum(a))*r_f

0.031